# 05 — Evaluation & Model Comparison
Top-1, top-5, confusion matrices, t-SNE embeddings, and attention maps across all 4 models.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))


In [ ]:
# Aggregate results from metrics.txt files
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np

models = ['vit', 'gat_vit', 'sparse_gat', 'deit']
labels = ['ViT-Small', 'GAT-ViT', 'Sparse-GAT-ViT', 'DeiT-Small']
top1_vals, top5_vals = [], []

for m in models:
    p = Path(f'../results/{m}/metrics.txt')
    if p.exists():
        data = dict(line.strip().split('=') for line in p.read_text().splitlines() if '=' in line)
        top1_vals.append(float(data.get('top1', 0)) * 100)
        top5_vals.append(float(data.get('top5', 0)) * 100)
    else:
        top1_vals.append(0)
        top5_vals.append(0)
        print(f'  [!] {m}/metrics.txt not found — run evaluate.py first')

print('Top-1:', dict(zip(labels, top1_vals)))
print('Top-5:', dict(zip(labels, top5_vals)))

In [ ]:
# Grouped bar chart: Top-1 and Top-5
x = np.arange(len(labels))
w = 0.35
fig, ax = plt.subplots(figsize=(10, 6))
b1 = ax.bar(x - w/2, top1_vals, w, label='Top-1', color='steelblue')
b5 = ax.bar(x + w/2, top5_vals, w, label='Top-5', color='coral')
ax.set_xticks(x)
ax.set_xticklabels(labels, fontsize=11)
ax.set_ylabel('Accuracy (%)')
ax.set_title('Model Comparison on CIFAR-100')
ax.legend()
ax.set_ylim(0, 100)
for bar in b1: ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.5,
                        f'{bar.get_height():.1f}', ha='center', fontsize=9)
for bar in b5: ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.5,
                        f'{bar.get_height():.1f}', ha='center', fontsize=9)
plt.tight_layout()
plt.savefig('../results/model_comparison_bar.png', dpi=150)
plt.show()

In [ ]:
# Display confusion matrices side by side
from IPython.display import Image, display
for m in models:
    p = f'../results/{m}/confusion_matrix.png'
    if Path(p).exists():
        print(f'\n--- {m} ---')
        display(Image(p, width=600))
    else:
        print(f'{m}: confusion matrix not found')

In [ ]:
# t-SNE (requires checkpoint)
# !python ../src/visualize.py --mode tsne --model sparse_gat \
#         --checkpoint ../results/sparse_gat/best.pt --out ../results
p = '../results/sparse_gat/tsne.png'
if Path(p).exists():
    display(Image(p, width=700))
else:
    print('Run visualize.py --mode tsne first.')

In [ ]:
# Attention maps (ViT)
# !python ../src/visualize.py --mode attention --model vit \
#         --checkpoint ../results/vit/best.pt --out ../results
p = '../results/vit/attention_maps.png'
if Path(p).exists():
    display(Image(p, width=700))
else:
    print('Run visualize.py --mode attention first.')